<a href="https://colab.research.google.com/github/Limeng-svg/Grounded-PPE-Safety-Copilot/blob/main/notebooks/03_yoloworld_prompt_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This round will fix the model, images, annotations, and evaluation parameters, only changing one PPE prompt, and will automatically generate comparison results and a download package.

In [1]:
%pip -q install "ultralytics==8.4.129" "git+https://github.com/ultralytics/CLIP.git@68dce32140994dfcb645a1320c4ebdc034fc19fd"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import tempfile, shutil, json

import yaml
import torch
import ultralytics
import pandas as pd
import matplotlib.pyplot as plt

drive.mount("/content/drive")

assert ultralytics.__version__ == "8.4.129", \
    "Please restart the Colab session and run this cell again."
assert torch.cuda.is_available(), \
    "Please set the hardware accelerator to T4 GPU."

print("GPU:", torch.cuda.get_device_name(0))

ROOT = Path(
    "/content/drive/MyDrive/Grounded-PPE-Safety-Copilot"
)
SOURCE = ROOT / "data/mini_eval"

CLASS_NAMES = [
    "person",
    "hard_hat",
    "safety_vest",
]

EXPECTED_COUNTS = Counter({
    0: 60,
    1: 53,
    2: 37,
})

images = sorted((SOURCE / "images").glob("*.jpg"))

expected_stems = {
    f"{group}_{i:02d}"
    for group in ["easy", "hard", "violation"]
    for i in range(1, 5)
}

assert len(images) == 12, "Expected 12 images but found fewer. Please check the data path and contents."
assert {p.stem for p in images} == expected_stems

# Copy data to a temporary Colab directory for faster read/write access;
# This will not modify the original images and labels in Google Drive.
LOCAL = Path(
    tempfile.mkdtemp(
        prefix="ppe_prompt_ablation_",
        dir="/content",
    )
)

for folder in ["images/val", "labels/val"]:
    (LOCAL / folder).mkdir(parents=True)

counts = Counter()

for image in images:
    label = SOURCE / "labels" / f"{image.stem}.txt"
    assert label.is_file(), f"Missing label file: {label.name}"

    for row in label.read_text(
        encoding="utf-8"
    ).splitlines():
        if row.strip():
            parts = row.split()
            assert len(parts) == 5, \
                f"Incorrect label format: {label.name}"
            counts[int(parts[0])] += 1

    shutil.copy2(
        image,
        LOCAL / "images/val" / image.name,
    )
    shutil.copy2(
        label,
        LOCAL / "labels/val" / label.name,
    )

assert counts == EXPECTED_COUNTS, \
    f"Label count mismatch: {counts}"

print("Data preparation complete: 12 images, 150 ground truth bounding boxes.")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Mounted at /content/drive
GPU: Tesla T4
Data preparation complete: 12 images, 150 ground truth bounding boxes.


In [3]:
from ultralytics import YOLOWorld
# A0 is the baseline
# H1,H2 only change the safty hat propmt
# V1,V2 only change the safety vest propmt

EXPERIMENTS = [
    {
        "id": "A0",
        "hard_hat": "hard hat",
        "safety_vest": "safety vest",
    },
    {
        "id": "H1",
        "hard_hat": "safety helmet",
        "safety_vest": "safety vest",
    },
    {
        "id": "H2",
        "hard_hat": "construction helmet",
        "safety_vest": "safety vest",
    },
    {
        "id": "V1",
        "hard_hat": "hard hat",
        "safety_vest": "high visibility vest",
    },
    {
        "id": "V2",
        "hard_hat": "hard hat",
        "safety_vest": "reflective safety vest",
    },
]

# Must matching 02 evaluation parameters
VAL_ARGS = dict(
    imgsz=640,
    batch=2,
    device=0,
    workers=0,
    conf=0.001,
    iou=0.5,
    max_det=300,
    rect=False,
    augment=False,
    quantize=None,
    agnostic_nms=False,
    plots=True,
    save_json=True,
    verbose=True,
)

RUN_ID = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%S_%fZ")

RUN_ROOT = (
    ROOT
    / "outputs"
    / "prompt_ablation"
    / RUN_ID
)

RUN_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

model = YOLOWorld("yolov8s-worldv2.pt")

# put the model into GPU before the first time generate the propmt features
model.to("cuda:0")

assert next(model.model.parameters()).device.type == "cuda"
print("Model is in the GPU!")

records = []

for experiment in EXPERIMENTS:
    prompts = [
        "person",
        experiment["hard_hat"],
        experiment["safety_vest"],
    ]

    # Label name of each group yaml and propmt should completely match
    yaml_path = LOCAL / f"{experiment['id']}.yaml"

    yaml_path.write_text(
        yaml.safe_dump(
            {
                "path": str(LOCAL),
                "train": None,
                "val": "images/val",
                "names": dict(enumerate(prompts)),
            },
            sort_keys=False,
        ),
        encoding="utf-8",
    )

    print("\n" + "=" * 60)
    print(
        f"Runing {experiment['id']}: {prompts}"
    )

    model.set_classes(prompts.copy())

    current_names = (
        list(model.names.values())
        if isinstance(model.names, dict)
        else list(model.names)
    )

    assert current_names == prompts, \
        f"Model and propmt not matching: {current_names}"

    metrics = model.val(
        data=str(yaml_path),
        split="val",
        project=str(RUN_ROOT),
        name=experiment["id"],
        exist_ok=False,
        **VAL_ARGS,
    )

    # Double-checked that the authenticator hasn't changed its prompts.
    metric_names = (
        list(metrics.names.values())
        if isinstance(metrics.names, dict)
        else list(metrics.names)
    )
    assert metric_names == prompts

    # Verified read 150 frames for real
    assert [
        int(x) for x in metrics.nt_per_class
    ] == [60, 53, 37]

    # The array line number of AP is able not to amount to label number, Therefore, explicit mapping is established.
    ap_rows = {
        int(class_id): row
        for row, class_id
        in enumerate(metrics.box.ap_class_index)
    }
    assert set(ap_rows) == {0, 1, 2}

    record = {
        "experiment": experiment["id"],
        "person_prompt": prompts[0],
        "hard_hat_prompt": prompts[1],
        "safety_vest_prompt": prompts[2],
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "run_dir": str(metrics.save_dir),
    }

    for class_id, canonical_name in enumerate(
        CLASS_NAMES
    ):
        row = ap_rows[class_id]

        record[
            f"{canonical_name}_AP50"
        ] = float(metrics.box.ap50[row])

        record[
            f"{canonical_name}_AP50_95"
        ] = float(metrics.box.ap[row])

    records.append(record)

    print(
        f"{experiment['id']} completed："
        f"mAP50={record['mAP50']:.2%}，"
        f"mAP50-95={record['mAP50_95']:.2%}"
    )

    progress = {
        "dataset_role":
            "development_only_not_final_test",
        "model": "yolov8s-worldv2.pt",
        "validation_settings": VAL_ARGS,
        "experiments_completed": records,
    }

    (
        RUN_ROOT
        / "prompt_ablation_progress.json"
    ).write_text(
        json.dumps(
            progress,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        ),
        encoding="utf-8",
    )

    del metrics
    torch.cuda.empty_cache()

print("\nAll 5 sets of experiments were completed.")
print("Result directory: ", RUN_ROOT)

Model is in the GPU!

Runing A0: ['person', 'hard hat', 'safety vest']


100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 117MiB/s]


Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s-worldv2 summary: 236 layers, 164,026,601 parameters, 151,277,313 gradients, 31.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3294.4±1752.2 MB/s, size: 1744.1 KB)
val: Scanning /content/ppe_prompt_ablation_xbmu4o0g/labels/val... 12 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12/12 189.7it/s 0.1s
val: New cache created: /content/ppe_prompt_ablation_xbmu4o0g/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.3it/s 2.6s
                   all         12        150      0.789      0.541      0.703      0.409
                person         12         60      0.891      0.983       0.99      0.654
              hard hat         10         53      0.969      0.586      0.857      0.465
           safety vest          9         37      0.508     0.0541      0.263      0.109
Speed: 0.5ms preprocess, 19.2ms